In [ ]:
import requests
import time
import random
import hashlib
import smtplib
import threading
import sys  # Pour sys.exit si besoin
from datetime import datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --- CONFIGURATION ---

URL_TO_CHECK = "https://www.marathondumedoc.com/inscription/"

# --- LISTES DE DIFFUSION ---
LISTE_VIP = ["petillion99@gmail.com"]
LISTE_STANDARD = ["quentinlevdso@gmail.com"]

# Configuration Email
EMAIL_SENDER = "pierre.elipse@gmail.com"
EMAIL_PASSWORD = "mifz iexy csrq xsjr"

MIN_INTERVAL = 50
MAX_INTERVAL = 70

# --- FONCTIONS ---

def get_content_hash(url):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            content = response.text
            return hashlib.md5(content.encode('utf-8')).hexdigest()
        else:
            print(f"Erreur HTTP : {response.status_code}")
            return None
    except Exception as e:
        print(f"Erreur lors de la récupération de la page : {e}")
        return None

def send_email_to_list(recipients_list, subject, message):
    if not recipients_list:
        return

    print(f"[EMAIL] Envoi de '{subject}' à {len(recipients_list)} personnes...")

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(EMAIL_SENDER, EMAIL_PASSWORD)

        for dest in recipients_list:
            msg = MIMEMultipart()
            msg['From'] = EMAIL_SENDER
            msg['To'] = dest
            msg['Subject'] = subject
            msg.attach(MIMEText(message, 'plain'))
            server.send_message(msg_email) # Correction: variable name was msg_email above, here msg

        server.quit()
        print("[EMAIL] SUCCÈS.")
    except Exception as e:
        print(f"[EMAIL] ERREUR CRITIQUE : {e}")

def vip_resend_loop(subject, message):
    print(">>> Démarrage du cycle VIP en arrière-plan...")
    for i in range(1, 9):
        time.sleep(120)
        print(f">>> Rappel VIP n°{i}/8...")
        send_email_to_list(LISTE_VIP, subject, message)
    print(">>> Fin cycle VIP.")

def main():
    print("------------------------------------------------")
    print(" DÉMARRAGE DU SCRIPT MARATHON ")
    print("------------------------------------------------")
    print(f"URL : {URL_TO_CHECK}")
    print(f"VIPs : {LISTE_VIP}")
    print(f"Standards : {LISTE_STANDARD}")
    print("------------------------------------------------")

    # Test de connexion Email au démarrage
    print("Test de connexion au serveur Email...")
    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(EMAIL_SENDER, EMAIL_PASSWORD)
        server.quit()
        print("-> Connexion Email OK !")
    except Exception as e:
        print(f"-> ERREUR CONNEXION EMAIL : {e}")
        print("Le script ne peut pas envoyer de mails. Vérifiez vos identifiants.")
        # On ne quitte pas forcément, pour tester le scraping, mais on avertit.

    send_email_to_list(LISTE_VIP, "[MARATHON] Démarrage", "Le script est lancé.")

    current_hash = get_content_hash(URL_TO_CHECK)

    if current_hash is None:
        print("Impossible de récupérer la page. Arrêt.")
        return

    print("Surveillance active. Attente du prochain cycle...")
    last_heartbeat_date = None

    while True:
        try:
            now = datetime.now()
            current_date = now.date()

            # Heartbeat
            if now.hour >= 9 and last_heartbeat_date != current_date:
                send_email_to_list(LISTE_VIP, "[MARATHON] Check 9h", "Tout va bien.")
                last_heartbeat_date = current_date

            # Scrapping
            sleep_time = random.uniform(MIN_INTERVAL, MAX_INTERVAL)
            print(f"[{now.strftime('%H:%M:%S')}] Attente de {int(sleep_time)}s avant vérification...")
            time.sleep(sleep_time)

            new_hash = get_content_hash(URL_TO_CHECK)

            if new_hash and new_hash != current_hash:
                print("!!! CHANGEMENT DÉTECTÉ !!!")
                alert_subject = "[MARATHON DU MEDOC] INSCRIPTIONS OUVERTES"
                alert_msg = f"La page d'inscription pour le marathon du médoc est ouverte ! Le lien est : {URL_TO_CHECK}"

                all_recipients = LISTE_VIP + LISTE_STANDARD
                send_email_to_list(all_recipients, alert_subject, alert_msg)

                threading.Thread(target=vip_resend_loop, args=(alert_subject, alert_msg)).start()
                current_hash = new_hash
            elif new_hash == current_hash:
                pass

        except KeyboardInterrupt:
            print("\nArrêt manuel du script.")
            break
        except Exception as e:
            print(f"Erreur dans la boucle principale : {e}")
            # On continue quand même au cas où ce serait une erreur réseau temporaire

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"ERREUR FATALE AU DÉMARRAGE : {e}")
        print("Appuyez sur Entrée pour fermer...")
        input()